# 02 · Learning from unlabeled data

**Owner:** Allen
**Report section:** Learning from unlabeled data

**Goal.** Can the 3,000 unlabeled passages help? Compare (a) the baseline, (b) TF-IDF vocabulary fit on train + unlabeled, (c) one-shot pseudo-labelling at several confidence thresholds, (d) multi-round self-training. Report validation AUROC and the number of pseudo-labels added per round.

Run with *Restart & Run All*. Set `QUICK = True` for a fast debug pass on the 2,000-row
starter file, `QUICK = False` for the numbers that go in the report. Every result uses the
shared split from `src.data.get_split()` (seed 42), so numbers are comparable across notebooks.

**Outputs:** `results/unlabeled.csv`, `results/unlabeled_selftrain_rounds.csv`, `figures/unlabeled_*.png`.

In [ ]:
# --- setup: make `src/` importable from anywhere (repo root, notebooks/, Kaggle/Colab clone) ---
import sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

QUICK = True  # True: 2,000-row starter file for fast debugging. False: full data (report numbers).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import SEED, set_seed, RESULTS_DIR, FIGURES_DIR
from src.data import get_split, load_test, load_unlabeled, load_train, get_cv_folds, xy
from src.features import build_features
from src.models import make_model
from src.evaluate import (auroc, cross_val_auroc, plot_roc, savefig, log_result,
                          save_predictions, predict_proba1)

set_seed(SEED)
SUFFIX = "_quick" if QUICK else ""   # keeps QUICK-mode outputs separate from real ones
from src.models import self_train, select_confident

In [ ]:
train_df, val_df = get_split(quick=QUICK)
test_df = load_test()
X_tr, y_tr = xy(train_df)
X_va, y_va = xy(val_df)
unl_df = load_unlabeled()

# Optionally treat test texts as unlabeled data too (transductive). Test is the shifted
# distribution we actually care about, so this may help more than unlabeled.csv alone.
USE_TEST_AS_UNLABELED = False
X_unl = list(unl_df["text"]) + (list(test_df["text"]) if USE_TEST_AS_UNLABELED else [])
print(f"unlabeled pool: {len(X_unl)} texts")

results = []
def record(method, val_auc, **extra):
    row = {"method": method, "val_auroc": val_auc, **extra}
    results.append(row)
    log_result("unlabeled", {"method": method, "quick": QUICK, "test_as_unlabeled": USE_TEST_AS_UNLABELED,
                             **{k: v for k, v in extra.items() if k in ("threshold", "balance", "rounds")}},
               {k: v for k, v in row.items() if k not in ("method", "threshold", "balance", "rounds")})
    print(f"{method:40s} val AUROC = {val_auc:.4f}  {extra}")

## (a) Baseline: labelled data only

In [ ]:
base = make_model("baseline").fit(X_tr, y_tr)
base_val = predict_proba1(base, X_va)
record("a_baseline", auroc(y_va, base_val))

p_unl = predict_proba1(base, X_unl)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(p_unl, bins=50)
ax.set(title="Baseline P(AI) on unlabeled data", xlabel="P(AI)", ylabel="count")
savefig(fig, "unlabeled_prob_hist" + SUFFIX)
print(f"predicted AI share on unlabeled: {(p_unl >= 0.5).mean():.3f}")
for t in (0.8, 0.9, 0.95, 0.99):
    print(f"  confident at {t}: {((p_unl >= t) | (p_unl <= 1 - t)).sum()} / {len(p_unl)}")

## (b) TF-IDF vocabulary/IDF fit on train + unlabeled

No labels are used. The vectorizers just see more text, so n-grams common in unlabeled data get vocabulary entries and IDF weights.

In [ ]:
vocab_model = make_model("baseline", extra_texts=X_unl).fit(X_tr, y_tr)
record("b_vocab_train+unlabeled", auroc(y_va, predict_proba1(vocab_model, X_va)))

## (c) One-shot pseudo-labelling at different confidence thresholds

Keep unlabeled texts with P(AI) ≥ t (label 1) or ≤ 1−t (label 0), add them to the training set, refit, and evaluate on the untouched validation split.

In [ ]:
THRESHOLDS = [0.8, 0.9, 0.95, 0.99]
for t in THRESHOLDS:
    for balance in ([False, True] if t == 0.9 else [False]):
        idx, lab = select_confident(p_unl, t, balance=balance)
        m = make_model("baseline").fit(list(X_tr) + [X_unl[i] for i in idx], np.r_[y_tr, lab])
        record(f"c_pseudo_t{t}{'_bal' if balance else ''}", auroc(y_va, predict_proba1(m, X_va)),
               threshold=t, balance=balance, rounds=1, n_added=len(idx),
               frac_ai_added=float(lab.mean()) if len(idx) else np.nan)

## (d) Multi-round self-training

Each round adds confident examples from the remaining pool (labels fixed once added) and refits.

In [ ]:
ROUNDS = 2 if QUICK else 5
histories = []
for t, balance in [(0.9, False), (0.9, True)]:
    _, hist, pseudo = self_train(make_model("baseline"), X_tr, y_tr, X_unl, threshold=t,
                                 rounds=ROUNDS, balance=balance, X_val=X_va, y_val=y_va)
    hist = hist.assign(threshold=t, balance=balance)
    histories.append(hist)
    best = hist.loc[hist["val_auroc"].idxmax()]
    record(f"d_selftrain_t{t}{'_bal' if balance else ''}", float(hist["val_auroc"].iloc[-1]),
           threshold=t, balance=balance, rounds=ROUNDS, n_added=int(hist["n_pseudo_total"].iloc[-1]),
           best_round=int(best["round"]), best_round_val_auroc=float(best["val_auroc"]))

rounds_df = pd.concat(histories, ignore_index=True)
rounds_df.to_csv(RESULTS_DIR / f"unlabeled_selftrain_rounds{SUFFIX}.csv", index=False)
rounds_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for (t, bal), h in rounds_df.groupby(["threshold", "balance"]):
    lab = f"t={t}{' balanced' if bal else ''}"
    axes[0].plot(h["round"], h["val_auroc"], marker="o", label=lab)
    axes[1].plot(h["round"], h["n_pseudo_total"], marker="o", label=lab)
axes[0].axhline(results[0]["val_auroc"], color="k", ls="--", lw=0.8, label="baseline")
axes[0].set(title="Validation AUROC by self-training round", xlabel="round", ylabel="AUROC")
axes[1].set(title="Cumulative pseudo-labels", xlabel="round", ylabel="# examples")
for ax in axes: ax.legend(fontsize=8)
savefig(fig, "unlabeled_selftrain" + SUFFIX)

## Summary

In [ ]:
summary = pd.DataFrame(results)
summary["delta_vs_baseline"] = summary["val_auroc"] - summary.loc[0, "val_auroc"]
summary.to_csv(RESULTS_DIR / f"unlabeled_summary{SUFFIX}.csv", index=False)
summary.sort_values("val_auroc", ascending=False)

## Caveats / next steps

* The validation split comes from the *train* distribution. Unlabeled data that resembles
  test may help on Kaggle without moving validation AUROC. Check against Aidan's
  test-like validation subset (`21_generalization`).
* TODO: try `USE_TEST_AS_UNLABELED = True`, and pseudo-label using the best model from
  `10_model_comparison` instead of the baseline.
* Final submissions can use `python predict.py --model <m> --vocab-unlabeled --pseudo-threshold 0.9 --pseudo-rounds k`.